---

## 📝 练习题

完成本章学习后，请尝试 [exercises.md](./exercises.md) 中的练习题来巩固知识。

**建议：**
- 先完成基础题，确保理解核心概念
- 进阶题帮助你深入掌握技巧
- 挑战题锻炼综合应用能力

💪 加油！实践是最好的学习方式！

# 2.2 自动微分（Autograd）

本 notebook 详细介绍 PyTorch 的自动微分功能，这是深度学习框架的核心

In [ ]:
import torch

## 1. 基本反向传播：backward()

自动微分的核心是 `backward()` 方法，它自动计算梯度

In [ ]:
# 基本反向传播示例
w = torch.tensor([1.], requires_grad=True)
x = torch.tensor([2.], requires_grad=True)

a = torch.add(w, x)      # a = w + x = 3
b = torch.add(w, 1)      # b = w + 1 = 2
y = torch.mul(a, b)      # y = a * b = 6

print(f"w: {w}")
print(f"x: {x}")
print(f"a = w + x = {a}")
print(f"b = w + 1 = {b}")
print(f"y = a * b = {y}")

# 反向传播
y.backward()

print(f"\n反向传播后:")
print(f"w.grad: {w.grad}")
print(f"x.grad: {x.grad}")
print("\n梯度计算:")
print("dy/dw = d(a*b)/dw = da/dw * b + a * db/dw = 1*2 + 3*1 = 5")
print("dy/dx = d(a*b)/dx = da/dx * b + a * db/dx = 1*2 + 3*0 = 2")

## 2. 计算图可视化

PyTorch 会自动构建计算图，记录每个操作

In [ ]:
# 查看计算图
w = torch.tensor([1.], requires_grad=True)
x = torch.tensor([2.], requires_grad=True)

a = w + x
b = w + 1
y = a * b

print("计算图节点:")
print(f"w.grad_fn: {w.grad_fn}  (叶子节点，无 grad_fn)")
print(f"x.grad_fn: {x.grad_fn}  (叶子节点，无 grad_fn)")
print(f"a.grad_fn: {a.grad_fn}")
print(f"b.grad_fn: {b.grad_fn}")
print(f"y.grad_fn: {y.grad_fn}")

print("\nis_leaf 属性:")
print(f"w.is_leaf: {w.is_leaf}")
print(f"x.is_leaf: {x.is_leaf}")
print(f"a.is_leaf: {a.is_leaf}")
print(f"y.is_leaf: {y.is_leaf}")

## 3. retain_graph 参数

默认情况下，反向传播后计算图会被释放。如果需要多次反向传播，需要设置 `retain_graph=True`

In [ ]:
# retain_graph=True: 保留计算图
w = torch.tensor([1.], requires_grad=True)
x = torch.tensor([2.], requires_grad=True)

a = torch.add(w, x)
b = torch.add(w, 1)
y = torch.mul(a, b)

# 第一次反向传播
y.backward(retain_graph=True)
print(f"第一次 w.grad: {w.grad}")

# 第二次反向传播（梯度会累加）
y.backward()
print(f"第二次 w.grad: {w.grad}  (梯度累加)")

print("\n注意: 如果不设置 retain_graph=True，第二次 backward() 会报错")

## 4. 梯度累加

**重要**: PyTorch 的梯度不会自动清零，需要手动清零

In [ ]:
# 梯度累加示例
w = torch.tensor([1.], requires_grad=True)
x = torch.tensor([2.], requires_grad=True)

print("梯度累加:")
for i in range(4):
    a = torch.add(w, x)
    b = torch.add(w, 1)
    y = torch.mul(a, b)
    
    y.backward(retain_graph=(i < 3))
    print(f"第{i+1}次 w.grad: {w.grad}")

print("\n正确做法: 每次反向传播后清零梯度")
w = torch.tensor([1.], requires_grad=True)
for i in range(3):
    a = torch.add(w, x)
    b = torch.add(w, 1)
    y = torch.mul(a, b)
    
    y.backward()
    print(f"第{i+1}次 w.grad: {w.grad}")
    w.grad.zero_()  # 清零梯度

## 5. gradient 参数（雅可比向量积）

当输出不是标量时，需要提供 `gradient` 参数

In [ ]:
# gradient 参数: 加权不同输出的梯度
w = torch.tensor([1.], requires_grad=True)
x = torch.tensor([2.], requires_grad=True)

a = torch.add(w, x)      # a = w + x = 3
b = torch.add(w, 1)      # b = w + 1 = 2

y0 = torch.mul(a, b)     # y0 = (x+w) * (w+1) = 6
y1 = torch.add(a, b)     # y1 = (x+w) + (w+1) = 5

loss = torch.cat([y0, y1], dim=0)  # [y0, y1]

print(f"y0: {y0}")
print(f"y1: {y1}")
print(f"loss: {loss}")

# 使用 gradient 加权
grad_tensors = torch.tensor([1., 2.])
loss.backward(gradient=grad_tensors)

print(f"\nw.grad: {w.grad}")
print(f"\n计算过程:")
print(f"  dy0/dw = 2w + x + 1 = 2*1 + 2 + 1 = 5")
print(f"  dy1/dw = 2")
print(f"  w.grad = 1 * (dy0/dw) + 2 * (dy1/dw) = 1 * 5 + 2 * 2 = 9")

## 6. torch.autograd.grad：计算梯度并返回

`torch.autograd.grad` 计算梯度但不累积到 `.grad` 属性

In [ ]:
# torch.autograd.grad: 计算梯度并返回
x = torch.tensor([3.], requires_grad=True)
y = torch.pow(x, 2)     # y = x^2

# 一阶导数：dy/dx = 2x = 6
grad_1 = torch.autograd.grad(y, x, create_graph=True)
print(f"一阶导数: {grad_1[0]}")
print(f"x.grad: {x.grad}  (没有累积到 .grad)")

# 二阶导数：d(dy/dx)/dx = d(2x)/dx = 2
grad_2 = torch.autograd.grad(grad_1[0], x)
print(f"\n二阶导数: {grad_2[0]}")

print("\n说明:")
print("- grad_1[0] 是一阶导数 2x = 6")
print("- grad_2[0] 是二阶导数 2")
print("- create_graph=True 允许计算高阶导数")

## 7. detach - 切断梯度传播

`detach()` 创建一个新张量，与原张量共享数据但不参与梯度计算

In [ ]:
# detach 的作用
w = torch.tensor([1.], requires_grad=True)
x = torch.tensor([2.], requires_grad=True)

a = torch.add(w, x)
b = torch.add(w, 1)
y = torch.mul(a, b)

print("detach 前:")
print(f"a.requires_grad: {a.requires_grad}")
print(f"a: {a}")

# detach 后
a_detach = a.detach()
print(f"\ndetach 后:")
print(f"a_detach.requires_grad: {a_detach.requires_grad}")
print(f"a_detach: {a_detach}")

# 修改 detach 后的张量会影响原张量（共享数据）
a_detach.data[0] = 999
print(f"\n修改 a_detach 后:")
print(f"a: {a}  (也被修改了)")
print(f"a_detach: {a_detach}")

print("\n说明: detach 创建的张量与原张量共享数据，但不参与梯度计算")

In [ ]:
# detach 在梯度传播中的作用
x = torch.tensor([1.], requires_grad=True)
y = x ** 2
z1 = y * 3  # 正常的梯度传播

z1.backward()
print(f"正常梯度传播: x.grad = {x.grad}")

# 使用 detach 切断梯度
x = torch.tensor([1.], requires_grad=True)
y = x ** 2
y_detach = y.detach()
z2 = y_detach * 3  # 梯度被切断

print(f"\ndetach 后: z2.requires_grad = {z2.requires_grad}")
print("说明: 由于 y_detach 不需要梯度，z2 也不需要梯度")

## 8. torch.no_grad() - 禁用梯度计算

在推理或评估时，使用 `torch.no_grad()` 可以节省内存和提高速度

In [ ]:
# torch.no_grad() 的作用
x = torch.tensor([1.], requires_grad=True)
y = x ** 2

# 不使用 no_grad：会构建计算图
z1 = y * 2
print(f"不使用 no_grad:")
print(f"z1.requires_grad: {z1.requires_grad}")
print(f"z1.grad_fn: {z1.grad_fn}")

# 使用 no_grad：不构建计算图
with torch.no_grad():
    z2 = y * 2
    print(f"\n使用 no_grad:")
    print(f"z2.requires_grad: {z2.requires_grad}")
    print(f"z2.grad_fn: {z2.grad_fn}")

print("\n使用场景:")
print("- 模型推理时")
print("- 评估模型性能时")
print("- 更新参数时（如 optimizer.step()）")

## 9. torch.set_grad_enabled() - 动态控制梯度

In [ ]:
# 动态控制是否计算梯度
def compute(x, requires_grad):
    with torch.set_grad_enabled(requires_grad):
        y = x ** 2
        return y

x = torch.tensor([1.], requires_grad=True)

# 训练模式
y_train = compute(x, requires_grad=True)
print(f"训练模式: y.requires_grad = {y_train.requires_grad}")

# 评估模式
y_eval = compute(x, requires_grad=False)
print(f"评估模式: y.requires_grad = {y_eval.requires_grad}")

## 10. 叶子节点和非叶子节点

- **叶子节点**: 用户创建的张量（`requires_grad=True`）
- **非叶子节点**: 运算产生的张量

In [ ]:
# 叶子节点 vs 非叶子节点
w = torch.tensor([1.], requires_grad=True)  # 叶子节点
x = torch.tensor([2.], requires_grad=True)  # 叶子节点

a = torch.add(w, x)  # 非叶子节点
b = torch.add(w, 1)  # 非叶子节点
y = torch.mul(a, b)  # 非叶子节点

print("叶子节点:")
print(f"w.is_leaf: {w.is_leaf}, w.requires_grad: {w.requires_grad}")
print(f"x.is_leaf: {x.is_leaf}, x.requires_grad: {x.requires_grad}")

print("\n非叶子节点:")
print(f"a.is_leaf: {a.is_leaf}, a.requires_grad: {a.requires_grad}")
print(f"b.is_leaf: {b.is_leaf}, b.requires_grad: {b.requires_grad}")
print(f"y.is_leaf: {y.is_leaf}, y.requires_grad: {y.requires_grad}")

y.backward()

print("\n反向传播后:")
print(f"w.grad: {w.grad}  (叶子节点保留梯度)")
print(f"x.grad: {x.grad}  (叶子节点保留梯度)")
print(f"a.grad: {a.grad}  (非叶子节点不保留梯度)")

## 11. 实际应用：简单神经网络的反向传播

In [ ]:
# 简单神经网络示例
import torch.nn.functional as F

# 输入和权重
x = torch.randn(1, 3)  # 1个样本，3个特征
w1 = torch.randn(3, 4, requires_grad=True)  # 第一层权重
b1 = torch.randn(1, 4, requires_grad=True)  # 第一层偏置
w2 = torch.randn(4, 2, requires_grad=True)  # 第二层权重
b2 = torch.randn(1, 2, requires_grad=True)  # 第二层偏置

# 前向传播
h = torch.matmul(x, w1) + b1  # 隐藏层
h = F.relu(h)  # 激活函数
y = torch.matmul(h, w2) + b2  # 输出层

# 损失函数（简单的均方误差）
target = torch.randn(1, 2)
loss = ((y - target) ** 2).mean()

print(f"输入形状: {x.shape}")
print(f"隐藏层形状: {h.shape}")
print(f"输出形状: {y.shape}")
print(f"\n损失: {loss}")

# 反向传播
loss.backward()

print("\n梯度:")
print(f"w1.grad 形状: {w1.grad.shape}")
print(f"b1.grad 形状: {b1.grad.shape}")
print(f"w2.grad 形状: {w2.grad.shape}")
print(f"b2.grad 形状: {b2.grad.shape}")

# 梯度下降更新（手动）
lr = 0.01
with torch.no_grad():
    w1 -= lr * w1.grad
    b1 -= lr * b1.grad
    w2 -= lr * w2.grad
    b2 -= lr * b2.grad
    
print("\n参数已更新（梯度下降）")

## 12. 总结

### 基本概念
- **requires_grad=True**: 告诉 PyTorch 需要计算梯度
- **backward()**: 反向传播，计算梯度
- **grad**: 存储梯度值
- **grad_fn**: 记录创建张量的运算

### 叶子节点
- 用户创建的张量是叶子节点
- 运算产生的是非叶子节点
- 只有叶子节点保留梯度

### 重要方法
- `backward(retain_graph=True)`: 保留计算图
- `torch.autograd.grad()`: 计算梯度并返回
- `detach()`: 切断梯度传播
- `torch.no_grad()`: 禁用梯度计算

### 关键要点
1. **梯度不会自动清零**，需要手动 `grad.zero_()`
2. **默认只有叶子节点保留梯度**
3. **反向传播后计算图被释放**（除非 `retain_graph=True`）
4. **推理时使用 `torch.no_grad()`** 节省内存
5. **detach 创建共享数据但不参与梯度计算的张量**

### 常见错误
- 忘记设置 `requires_grad=True`
- 忘记清零梯度
- 对非标量调用 `backward()` 时忘记提供 `gradient` 参数
- 在需要多次反向传播时忘记 `retain_graph=True`

---

## 📝 练习题

完成本章学习后，请尝试 [exercises.md](./exercises.md) 中的练习题来巩固知识。

**建议：**
- 先完成基础题，确保理解核心概念
- 进阶题帮助你深入掌握技巧
- 挑战题锻炼综合应用能力

💪 加油！实践是最好的学习方式！